# 应用组织与配置

学习目标：把相关路由组合成小型 Python 包，校验环境配置，并按不同配置创建独立应用实例。

前置知识：Python 模块与包、类型标注、嵌套函数、FastAPI 路由、环境变量、异常处理与接口测试。

运行环境：Python 3.12、FastAPI 0.141.1、Pydantic v2、pydantic-settings 2.15.0。

环境准备：见 [FastAPI 环境与运行说明](README.md)。

工作目录：content/Web与应用开发/FastAPI。选择课程环境的 Python 3 (ipykernel)，从空内核顺序运行。参数和配置实验使用 TestClient；末节通过本地 8080 端口运行配套包，完成后关闭服务。

配套脚本：位于 scripts/08-application-and-settings/study_api。

1. [\_\_init\_\_.py](scripts/08-application-and-settings/study_api/__init__.py)：标识本章的普通 Python 包。
2. [routes.py](scripts/08-application-and-settings/study_api/routes.py)：保存两条记录路由。
3. [config.py](scripts/08-application-and-settings/study_api/config.py)：保存配置类型与校验条件。
4. [main.py](scripts/08-application-and-settings/study_api/main.py)：提供创建应用的函数，供客户端与 Uvicorn 调用。

## 1 用 APIRouter 收集两条路由

APIRouter 用来把一组路径操作放在一起。定义路由时使用 router.get；要让请求找到这些路由，再通过应用的 include_router 加入它们。

先定义两个小接口：一个返回固定列表，一个回显整数编号。它们只用于观察路由组织，没有数据库查询。

In [1]:
from fastapi import APIRouter

router = APIRouter()


@router.get("/records")
def list_records() -> list[dict[str, str]]:
    return [{"title": "练习路由"}, {"title": "整理配置"}]


@router.get("/records/{record_id}")
def read_record(record_id: int) -> dict[str, int]:
    return {"record_id": record_id}

现在把 router 加入一个 FastAPI 应用，并分别调用两条路径。TestClient 直接调用应用，with 负责结束客户端的使用。

In [2]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI()
app.include_router(router)

with TestClient(app) as client:
    records = client.get("/records")
    record = client.get("/records/7")
assert records.status_code == 200
assert len(records.json()) == 2
assert record.json() == {"record_id": 7}
# 两个接口已经加入 app；APIRouter 本身负责收集路由。
print(records.json())  # 预期：[{'title': '练习路由'}, {'title': '整理配置'}]。
print(record.json())  # 预期：{'record_id': 7}。

[{'title': '练习路由'}, {'title': '整理配置'}]
{'record_id': 7}


C:\Users\ZHUANG\miniconda3\envs\hands-on-computing\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## 2 加入公共前缀和文档分组

include_router 的 prefix 会加在这组路由的路径前面；tags 会给每个路径操作补充文档标签。前缀改变请求地址，标签用于 OpenAPI 和交互文档的分组。

下面复用同一个 router，加入新的 grouped_app：/records 变成 /api/records。非空 prefix 以斜杠开头、末尾不带斜杠；每条路由自己的路径以斜杠开头。

In [3]:
grouped_app = FastAPI()
grouped_app.include_router(router, prefix="/api", tags=["records"])

with TestClient(grouped_app) as client:
    response = client.get("/api/records/7")
    old_path = client.get("/records/7")
operation = grouped_app.openapi()["paths"]["/api/records/{record_id}"]["get"]
assert response.json() == {"record_id": 7}
assert old_path.status_code == 404
assert operation["tags"] == ["records"]
# 新应用只包含带前缀的地址；records 是文档标签。
print(response.status_code, response.json(), old_path.status_code)  # 预期：200 {'record_id': 7} 404。
print("文档标签：", operation["tags"])  # 预期：文档标签： ['records']。

200 {'record_id': 7} 404
文档标签： ['records']


## 3 将已经明确的路由移入一个模块

需要从不同入口导入这组路由时，再把第 1 节的代码放到 routes.py。当前只需要一个普通包和一个路由模块，文件位置如下；\_\_init\_\_.py 标识普通包，routes.py 导出 router。

```text
scripts/08-application-and-settings/
└── study_api/
    ├── __init__.py
    └── routes.py
```

配套 routes.py 保存的就是前面已经运行的两条路由。Notebook 仍以课程目录为工作目录，所以导入前临时把包的父目录放进 sys.path（模块搜索路径），随后恢复。路径都相对于课程目录。

In [4]:
from pathlib import Path
import sys

package_parent = Path("scripts/08-application-and-settings").resolve()
original_search_path = sys.path.copy()
try:
    sys.path.insert(0, str(package_parent))
    from study_api.routes import router as packaged_router
finally:
    sys.path[:] = original_search_path

module_app = FastAPI()
module_app.include_router(packaged_router, prefix="/api", tags=["records"])
with TestClient(module_app) as client:
    response = client.get("/api/records/8")
# 路由换到模块中，调用地址和参数解析保持相同。
assert response.json() == {"record_id": 8}
print(response.json())  # 预期：{'record_id': 8}。

{'record_id': 8}


## 4 用 Settings 声明配置类型和范围

应用名称、版本序号等值可以作为配置提供。环境变量中保存的是字符串；pydantic-settings 的 BaseSettings 会读取输入并按字段类型转换、校验。

SettingsConfigDict 的 env_prefix 指定环境变量前缀。例如 revision 对应 HOC_FASTAPI08_REVISION。默认配置下名称不区分大小写，这里统一用大写环境变量名。

本例自行约定 revision 是 1～99 的整数版本序号。Settings 的字段默认值用于没有提供对应配置时；Field 的约束也会作用于配置输入。

In [5]:
from pydantic import Field
from pydantic_settings import BaseSettings, SettingsConfigDict


class Settings(BaseSettings):
    model_config = SettingsConfigDict(env_prefix="HOC_FASTAPI08_")

    app_name: str = "学习记录 API"
    revision: int = Field(default=1, ge=1, le=99)

先直接传入两个配置值，观察构造出的对象。这些值是本章的公开显示信息。

In [6]:
explicit_settings = Settings(app_name="课堂 API", revision=2)
assert explicit_settings.app_name == "课堂 API"
assert explicit_settings.revision == 2
# 配置对象的字段已经具有声明的类型。
print(explicit_settings.model_dump())  # 预期：{'app_name': '课堂 API', 'revision': 2}。
print(type(explicit_settings.revision).__name__)  # 预期：int。

{'app_name': '课堂 API', 'revision': 2}
int


## 5 从环境读取配置，并拒绝非法值

本例没有启用命令行解析或自定义配置来源。对当前使用的来源，同名字段依次优先采用：构造时传入的参数、环境变量、字段默认值。

用 os.environ 临时设置当前进程的环境变量，比较环境提供的 revision 与显式参数。实验结束时用 finally 恢复原值，原先不存在则删除，避免影响后续操作。

In [7]:
import os

variable = "HOC_FASTAPI08_REVISION"
previous_value = os.environ.get(variable)
try:
    os.environ[variable] = "3"
    from_environment = Settings(app_name="环境配置 API")
    overridden = Settings(app_name="显式配置 API", revision=4)
    assert from_environment.revision == 3
    assert overridden.revision == 4
    # 环境中的字符串 3 转成整数；显式传入的 4 优先。
    print(from_environment.revision, type(from_environment.revision).__name__)  # 预期：3 int。
    print("显式参数：", overridden.revision)  # 预期：4，显式传入值覆盖环境中的 3。
finally:
    if previous_value is None:
        os.environ.pop(variable, None)
    else:
        os.environ[variable] = previous_value
assert os.environ.get(variable) == previous_value

3 int
显式参数： 4


配置在创建 `Settings` 对象时校验。下面分别观察超出范围与无法转换成整数的原始 `ValidationError`；每个单元的 `finally` 只负责恢复环境，不捕获或替换异常。

In [8]:
previous_value = os.environ.get(variable)
try:
    os.environ[variable] = "0"
    Settings(app_name="错误配置 API")  # 预期：ValidationError，revision 的 greater_than_equal。
finally:
    if previous_value is None:
        os.environ.pop(variable, None)
    else:
        os.environ[variable] = previous_value

ValidationError: 1 validation error for Settings
revision
  Input should be greater than or equal to 1 [type=greater_than_equal, input_value='0', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than_equal

`many` 无法转换成整数，错误类别为 `int_parsing`；上一例的 `0` 可以转换但违反最小值，错误类别为 `greater_than_equal`。

In [9]:
previous_value = os.environ.get(variable)
try:
    os.environ[variable] = "many"
    Settings(app_name="错误配置 API")  # 预期：ValidationError，revision 的 int_parsing。
finally:
    if previous_value is None:
        os.environ.pop(variable, None)
    else:
        os.environ[variable] = previous_value

ValidationError: 1 validation error for Settings
revision
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='many', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing

## 6 用 create_app 按配置创建应用

将创建 FastAPI、加入路由和设置应用信息的动作放进 create_app。每次调用都会返回一个新的应用对象，这样的函数通常称为应用工厂（application factory）。

下面的函数接收可选 Settings；没有传入时才读取环境并创建配置。嵌套的 read_info 使用本次调用中的 settings，返回两个公开配置字段。

In [10]:
def create_app(settings: Settings | None = None) -> FastAPI:
    if settings is None:
        settings = Settings()
    application = FastAPI(title=settings.app_name)
    application.include_router(router, prefix="/api", tags=["records"])

    @application.get("/info", tags=["application"])
    def read_info() -> dict[str, str | int]:
        return {"app_name": settings.app_name, "revision": settings.revision}

    return application

创建两个应用，分别传入自己的配置。路由定义相同，但应用对象、标题和配置结果分别确定。

In [11]:
first_app = create_app(Settings(app_name="甲应用", revision=1))
second_app = create_app(Settings(app_name="乙应用", revision=2))

with TestClient(first_app) as client:
    first_info = client.get("/info").json()
with TestClient(second_app) as client:
    second_info = client.get("/info").json()
assert first_app is not second_app
assert first_app.title == "甲应用" and second_app.title == "乙应用"
assert first_info == {"app_name": "甲应用", "revision": 1}
assert second_info == {"app_name": "乙应用", "revision": 2}
print(first_info)  # 预期：{'app_name': '甲应用', 'revision': 1}。
print(second_info)  # 预期：{'app_name': '乙应用', 'revision': 2}。

{'app_name': '甲应用', 'revision': 1}
{'app_name': '乙应用', 'revision': 2}


创建后的 Settings 对象不会持续监听环境变量。需要用新的环境值创建应用时，应再次调用工厂；本例把配置读取安排在创建应用的阶段。

In [12]:
previous_value = os.environ.get(variable)
try:
    os.environ[variable] = "5"
    earlier = Settings(app_name="早先读取")
    os.environ[variable] = "6"
    later_app = create_app(Settings(app_name="后来读取"))
    with TestClient(later_app) as client:
        later_info = client.get("/info").json()
    # 旧配置保留 5；新创建的配置读取到 6。
    assert earlier.revision == 5
    assert later_info["revision"] == 6
    print(earlier.revision, later_info)  # 预期：5 {'app_name': '后来读取', 'revision': 6}。
finally:
    if previous_value is None:
        os.environ.pop(variable, None)
    else:
        os.environ[variable] = previous_value
assert os.environ.get(variable) == previous_value

5 {'app_name': '后来读取', 'revision': 6}


## 7 把配置和工厂放入包，再由 Uvicorn 启动

现在再增加两个文件：config.py 保存第 4 节的 Settings 定义；main.py 保存第 6 节的 create_app，并在文件开头导入所需名称。一个点表示从当前包导入。

```python
from fastapi import FastAPI

from .config import Settings
from .routes import router
```

完整配套包只有这四个文件，没有在导入时调用 create_app 或启动服务器。Python 导入会执行模块顶层语句，因此读取配置和启动服务的时机需要在代码中明确安排。

```text
scripts/08-application-and-settings/
└── study_api/
    ├── __init__.py
    ├── routes.py
    ├── config.py
    └── main.py
```

前面已导入 study_api 包，下面从包中取得工厂，并创建一个配置明确的应用进行调用。

In [13]:
from study_api.config import Settings as PackageSettings
from study_api.main import create_app as package_create_app

packaged_app = package_create_app(PackageSettings(app_name="包内应用", revision=7))
with TestClient(packaged_app) as client:
    info = client.get("/info")
    record = client.get("/api/records/8")
assert info.json() == {"app_name": "包内应用", "revision": 7}
assert record.json() == {"record_id": 8}
print(info.json(), record.json())  # 预期：{'app_name': '包内应用', 'revision': 7} {'record_id': 8}。

{'app_name': '包内应用', 'revision': 7} {'record_id': 8}


再让一个独立 Python 进程只做导入并退出，避免已有的模块缓存影响观察。subprocess.run 的 cwd 指定包的父目录，timeout 限制等待时间；sys.executable 使用当前内核对应的 Python。

子进程收到一个故意非法的 revision。导入阶段只提供函数与类型，尚未实例化配置，因此导入仍能正常完成。

In [14]:
import subprocess

import_only = "from study_api.main import create_app\nprint(callable(create_app))"
result = subprocess.run(
    [sys.executable, "-B", "-c", import_only],
    cwd="scripts/08-application-and-settings",
    env={**os.environ, "HOC_FASTAPI08_REVISION": "invalid"},
    capture_output=True,
    text=True,
    check=True,
    timeout=15,
)
# 正常退出且打印 True：本包的导入没有调用工厂或运行前台服务器。
# env 只传给子进程，当前进程的环境没有被修改。
assert result.stdout.strip() == "True"
print("导入进程退出码：", result.returncode, "工厂可调用：", result.stdout.strip())  # 预期：导入进程退出码： 0 工厂可调用： True。

导入进程退出码： 0 工厂可调用： True


真正提供 HTTP 服务时，由 Uvicorn 导入并调用工厂。下面命令从课程目录执行；study_api.main 是模块，冒号后的 create_app 是可调用对象，--factory 告诉 Uvicorn 调用它取得应用，--app-dir 指定包的父目录。

Step 1：在已激活环境的独立终端启动本地服务。

```bash
python -m uvicorn study_api.main:create_app --factory --app-dir scripts/08-application-and-settings --host 127.0.0.1 --port 8080
```

Step 2：浏览器打开 http://127.0.0.1:8080/docs，展开 records 分组下的 GET /api/records/{record_id}，点击 Try it out，输入 8 后点击 Execute；响应应为 200，响应体包含 record_id: 8。

Step 3：在 application 分组调用 GET /info，检查应用名称和版本序号；未设置本章环境变量时，分别为“学习记录 API”和 1。

Step 4：回到服务终端按 Ctrl+C，等待 Uvicorn 显示关闭完成。此时再次访问本地接口应连接失败。

## 本章小结

（1）APIRouter 收集相关路径操作，include_router 将它们加入应用；prefix 改变路径，tags 用于文档分组。

（2）先确定模块职责，再拆文件。本章只把路由、配置和应用创建放到各自模块。

（3）BaseSettings 在创建配置对象时读取并校验环境输入；本例的显式参数优先于环境变量，修改实验环境后要恢复。

（4）create_app 返回新应用；导入模块、创建应用、启动服务器是代码中分别安排的动作。

自查：把 revision 改为非法值时，哪个动作会失败？如果只改变文档标签，请求地址会不会随之改变？

## 练习

（1）把 grouped_app 的前缀改为 /v1，标签改为 notes。验证标准：/v1/records/7 返回 200，/api/records/7 返回 404，OpenAPI 中的标签为 notes。

In [15]:
# 在此完成本题；按题目条件核对结果。

（2）在 Settings 中增加整数字段 max_records，默认 10，范围为 1～50。验证标准：环境变量 HOC_FASTAPI08_MAX_RECORDS=20 得到整数 20，设为 0 抛出 ValidationError；实验结束后恢复原环境。

In [16]:
# 在此完成本题；按题目条件核对结果。

（3）在包的 routes.py 中增加 GET /summary，返回固定 total: 2。验证标准：工厂创建的应用能通过 /api/summary 返回 200；配置模块与工厂函数不用增加业务实现。

In [17]:
# 在此完成本题；按题目条件核对结果。

（4）保持导入进程中的非法 revision，再把执行内容改为导入后调用 create_app()。验证标准：单纯导入仍成功，调用工厂时配置校验失败；不要启动 Uvicorn 来验证这个配置错误。

In [18]:
# 在此完成本题；按题目条件核对结果。

### 第 4 题提示与解析

提示 1：保留子进程 `env` 中非法的 `HOC_FASTAPI08_REVISION`；只把 `import_only` 的内容增加一行 `create_app()`。

提示 2：这次失败是观察目标；用 `check=False` 取得返回码和 `stderr`，核对配置字段 `revision` 的校验错误。

解析：单纯导入只定义函数和类型，返回码为 0；调用工厂才执行 `Settings()`，字符串 `invalid` 无法转换成整数，因此退出码非零且诊断包含 `ValidationError`、`revision` 和 `int_parsing`。无需启动服务器，配置问题已经在创建应用时暴露。

修改配套模块后重启 Notebook 内核再顺序运行，避免使用已导入的旧模块。第 2 题恢复环境的 `finally` 用于进程状态清理。

## 参考与引用来源

- **FastAPI 官方文档（fastapi.tiangolo.com）**：[Bigger Applications](https://fastapi.tiangolo.com/tutorial/bigger-applications/)，定位 APIRouter、How relative imports work、Include an APIRouter with a custom prefix, tags, responses, and dependencies，支持路由组合、前缀、文档标签和包内导入；[Settings and Environment Variables](https://fastapi.tiangolo.com/advanced/settings/)，定位 Types and validation、Create the Settings object、Settings in another module，支持配置类型、字段校验及拆分配置模块；[Testing](https://fastapi.tiangolo.com/tutorial/testing/#using-testclient)，支持应用内调用与断言。
- **Pydantic 官方文档（pydantic.dev）**：[Settings Management](https://pydantic.dev/docs/validation/latest/concepts/pydantic_settings/)，定位 Usage、Environment variable names、Case-sensitivity、Parsing environment variable values、Field value priority 及 In-place reloading，支持 BaseSettings、env_prefix、类型转换、默认来源优先级和配置重新读取的条件。
- **Python 3.12 官方文档（docs.python.org）**：[Modules](https://docs.python.org/3.12/tutorial/modules.html#packages)，定位 More on Modules、The Module Search Path 与 Packages，支持导入时执行模块代码、模块缓存与普通包；[sys.path](https://docs.python.org/3.12/library/sys.html#sys.path)，支持临时设置模块搜索路径；[os.environ](https://docs.python.org/3.12/library/os.html#os.environ)，支持读取、修改和恢复当前进程环境；[subprocess.run](https://docs.python.org/3.12/library/subprocess.html#subprocess.run)，支持独立进程、cwd、env、超时和退出码检查。
- **Uvicorn 官方文档（uvicorn.dev）**：[Settings](https://uvicorn.dev/settings/#application)，定位 Application 和 Socket Binding，支持应用工厂、模块导入路径、host 与 port 参数。
- **Starlette 官方文档（starlette.dev）**：[TestClient](https://starlette.dev/testclient/)，支持本章客户端的 with 使用范围。